# Data Vortex — Phase 2: SQL Challenge 2
## Top Creator Audience Leaderboard

### 1. Challenge Description
Construct a creator audience leaderboard ranking users by their registered follower count.
For each qualifying creator who has authored at least one post, display:
1. `user_id`: Unique creator primary key.
2. `location`: Metropolitan city and country.
3. `language`: Primary language code.
4. `follower_count`: Registered follower base.
5. `post_count`: Total posts authored (`COUNT(p.post_id)`).
6. `avg_likes`: Mean likes per post (`ROUND(AVG(p.likes), 2)`).
7. `avg_shares`: Mean shares per post (`ROUND(AVG(p.shares), 2)`).
8. `avg_comments`: Mean comments per post (`ROUND(AVG(p.comments), 2)`).

Restricted to the top 20 creators sorted by `follower_count DESC`.

In [ ]:
import os
import sqlite3
import pandas as pd

# Database Path
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_02_top_creator_audience.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Query File:  {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to database successfully.")

### 2. SQL Query Execution
Execute the leaderboard query from `sql/challenge_02_top_creator_audience.sql`.

In [ ]:
with open(SQL_PATH, "r", encoding="utf-8") as f:
    query = f.read()

print("=== SQL Query ===")
print(query)

# Execute and render top 20
df_top20 = pd.read_sql(query, conn)
df_top20

### 3. Verification & Extremes Across Entire Database
Identify the users with the highest follower count, highest post volume, and highest average likes across the entire 1,500-user database.

In [ ]:
# Query overall database extremes
q_extremes = """
WITH user_summary AS (
    SELECT 
        u.user_id,
        u.location,
        u.language,
        u.follower_count,
        COUNT(p.post_id) AS post_count,
        ROUND(AVG(p.likes), 2) AS avg_likes
    FROM users u
    JOIN posts p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.location, u.language, u.follower_count
)
SELECT 'Highest Followers' AS extreme_type, * FROM user_summary ORDER BY follower_count DESC LIMIT 1
UNION ALL
SELECT 'Highest Post Count' AS extreme_type, * FROM user_summary ORDER BY post_count DESC LIMIT 1
UNION ALL
SELECT 'Highest Avg Likes' AS extreme_type, * FROM user_summary ORDER BY avg_likes DESC LIMIT 1;
"""
df_extremes = pd.read_sql(q_extremes, conn)
df_extremes

### 4. Analytical Findings & Interpretation
- **Highest-Follower User:** `user_3o7w66o2` with **49,944 followers** (Berlin, Germany; 8 posts; avg likes 3,394.00).
- **Highest-Post-Count User:** `user_zqv2vrf5` with **22 posts** (13,531 followers; San Jose, USA).
- **Highest-Average-Likes User:** `user_uw0vd87k` with **4,821.33 avg likes** (11,640 followers; Milan, Italy).
- **Do Highest Followers Correspond to Highest Engagement?** **No.** The empirical data demonstrates that having more followers does not correspond to higher engagement. In the top 20 follower group, average likes range widely from 1,600.33 to 3,394.00, and the platform-wide highest average likes belongs to a user in the bottom quartile of followers.

In [ ]:
# Close database connection
conn.close()
print("Database connection closed cleanly.")